In [1]:
#import libraries
import pandas as pd
import numpy as np

#inputs are fighter stats, outputs are fighter won or lost
#given fighters history, what will the outcome be for a fight?

#input tables: fighter fight stats, round stats, fights
#output: fighter x has y probability of winning

#read csv first.
csv_path = "../out.csv" 
df = pd.read_csv(csv_path)
df.info


<bound method DataFrame.info of        event_id  event_date  fight_id  fighter_id         fighter_name  \
0           792  1994-03-11      8894        2476         Royce Gracie   
1           792  1994-03-11      8894        2719        Patrick Smith   
2           792  1994-03-11      8895        2476         Royce Gracie   
3           792  1994-03-11      8895        2711        Remco Pardoel   
4           792  1994-03-11      8896        2719        Patrick Smith   
...         ...         ...       ...         ...                  ...   
17865         1  2026-10-10        10          20            RJ Harris   
17866         1  2026-10-10        11          21        Alice Pereira   
17867         1  2026-10-10        11          22  Daria Zhelezniakova   
17868         1  2026-10-10        12          23   Ernesta Kareckaite   
17869         1  2026-10-10        12          24        Melissa Gatto   

                      weight_class outcome  winner_id  knockdowns  \
0      UFC

In [2]:
#rename stats
rename_stats = {
    "knockdowns": "kd",
    "sig_strikes_landed": "ss_l",
    "sig_strikes_attempted": "ss_a",
    "takedowns_landed": "td_l",
    "takedowns_attempted": "td_a",
    "control_seconds": "ctrl_s",
}
df = pd.read_csv(csv_path, parse_dates=["event_date"]).rename(columns=rename_stats)
assert not df.duplicated(["fight_id", "fighter_id"]).any()
assert df.groupby("fight_id")["fighter_id"].nunique().eq(2).all()

df = df.sort_values(["fighter_id", "event_date", "fight_id"]).reset_index(drop=True)

In [3]:
#calculate historical win rates
#find all fights where winner_id == curr_fighter_id
#get fights that are win/loss not draw/nc
df["decisive"] = df["outcome"].eq("winner")
df["won"] = df["decisive"] & df["fighter_id"].eq(df["winner_id"])
agr = df.groupby(["fighter_id", "event_date"], as_index=False).agg(
    wins=("won", "sum"),
    decisive_fights=("decisive", "sum")
).sort_values(["fighter_id", "event_date"])
agr

,fighter_id,event_date,wins,decisive_fights
0,1,2019-10-18,1,1
1,1,2020-02-29,1,1
2,1,2020-06-27,1,1
3,1,2020-11-14,0,1
4,1,2021-04-24,1,1
...,...,...,...,...
17776,2762,1995-09-08,1,1
17777,2763,1995-07-14,0,1
17778,2764,1995-07-14,0,1
17779,2765,1994-12-16,0,1


In [4]:
#pre fight totals
grouped = agr.groupby("fighter_id")
#sum all wins divided by decisive fights and shift up one
for col in ["wins", "decisive_fights"]:
    agr[f"prior_{col}"] = grouped[col].transform(lambda x : x.cumsum().shift(fill_value=0))


agr["prior_wr"] = agr["prior_wins"] / agr["prior_decisive_fights"].replace(0, np.nan)
agr.head(10)


,fighter_id,event_date,wins,decisive_fights,prior_wins,prior_decisive_fights,prior_wr
0,1,2019-10-18,1,1,0,0,NaN
1,1,2020-02-29,1,1,1,1,1.000000
2,1,2020-06-27,1,1,2,2,1.000000
3,1,2020-11-14,0,1,3,3,1.000000
4,1,2021-04-24,1,1,3,4,0.750000
5,1,2021-07-24,1,1,4,5,0.800000
6,1,2021-12-04,0,1,5,6,0.833333
7,1,2022-02-05,1,1,5,7,0.714286
8,1,2022-06-11,1,1,6,8,0.750000
9,1,2022-10-01,1,1,7,9,0.777778


In [5]:
prior_cols = ["fighter_id", "event_date", "prior_wins", "prior_decisive_fights", "prior_wr"]
df = df.merge(agr[prior_cols], on=["fighter_id", "event_date"], how="left", validate="many_to_one")
df.sort_values(["fighter_id", "event_date"])
df

,event_id,event_date,fight_id,fighter_id,fighter_name,weight_class,outcome,winner_id,kd,ss_l,ss_a,td_l,td_a,ctrl_s,decisive,won,prior_wins,prior_decisive_fights,prior_wr
0,296,2019-10-18,3570,1,Brendan Allen,Middleweight Bout,winner,1.0,0.0,10.0,22.0,2.0,3.0,174.0,True,True,0,0,NaN
1,283,2020-02-29,3407,1,Brendan Allen,Middleweight Bout,winner,1.0,0.0,20.0,29.0,0.0,1.0,253.0,True,True,1,1,1.00
2,273,2020-06-27,3296,1,Brendan Allen,Middleweight Bout,winner,1.0,1.0,41.0,66.0,0.0,0.0,337.0,True,True,2,2,1.00
3,253,2020-11-14,3074,1,Brendan Allen,Catch Weight Bout,winner,438.0,0.0,51.0,90.0,0.0,0.0,0.0,True,False,3,3,1.00
4,234,2021-04-24,2864,1,Brendan Allen,Middleweight Bout,winner,1.0,0.0,9.0,16.0,2.0,2.0,196.0,True,True,3,4,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17865,787,1995-09-08,8931,2762,Joel Sutton,Open Weight Bout,winner,2762.0,NaN,NaN,NaN,NaN,NaN,NaN,True,True,1,1,1.00
17866,788,1995-07-14,8932,2763,He-Man Gipson,Open Weight Bout,winner,2718.0,NaN,NaN,NaN,NaN,NaN,NaN,True,False,0,0,NaN
17867,788,1995-07-14,8933,2764,Jack McGlaughlin,Open Weight Bout,winner,2762.0,NaN,NaN,NaN,NaN,NaN,NaN,True,False,0,0,NaN
17868,790,1994-12-16,8934,2765,Eldo Xavier Dias,Open Weight Bout,winner,2728.0,NaN,NaN,NaN,NaN,NaN,NaN,True,False,0,0,NaN


In [6]:
#historical avg stats
stats_cols = ["kd", "ss_l", "ss_a", "td_l", "td_a", "ctrl_s"]
#in earlier fights, fighters fought multiple times in one event
agr = df.groupby(["fighter_id", "event_date"], as_index=False).agg(
    ss_a=("ss_a", "sum"),
    ss_l=("ss_l", "sum"),
    td_a=("td_a", "sum"),
    td_l=("td_l", "sum"),
    kd=("kd", "sum"),
    kd_count=("kd", "count"),
    ctrl_s=("ctrl_s", "sum"),
    wins=("won", "sum"),
    decisive_fights=("decisive", "sum")
).sort_values(["fighter_id", "event_date"])

grouped = agr.groupby(["fighter_id"])
#create prior columns
for col in stats_cols + ["kd_count"]:
    agr[f"prior_{col}"] = grouped[col].transform(lambda x : x.cumsum().shift(fill_value=0))

prior_cols = [f"prior_{col}" for col in stats_cols + ["kd_count"]]
# grouped = df.groupby(["fighter_id, event_date"])
merge_keys = ["fighter_id", "event_date"]
df = df.drop(columns=prior_cols, errors="ignore").merge(agr[merge_keys + prior_cols], on=merge_keys, how='left', validate="many_to_one")



In [7]:
df[merge_keys + stats_cols + prior_cols + ["prior_wins"] + ["prior_wr"]]
#stats columns
print(df.select_dtypes(include="number").columns.to_list())

['event_id', 'fight_id', 'fighter_id', 'winner_id', 'kd', 'ss_l', 'ss_a', 'td_l', 'td_a', 'ctrl_s', 'prior_wins', 'prior_decisive_fights', 'prior_wr', 'prior_kd', 'prior_ss_l', 'prior_ss_a', 'prior_td_l', 'prior_td_a', 'prior_ctrl_s', 'prior_kd_count']


In [8]:
#add combined stats 
df["ss%"] = df["prior_ss_l"] / df["prior_ss_a"].replace(0, np.nan)
df["td%"] = df["prior_td_l"] / df["prior_td_a"].replace(0, np.nan)
df["kd_avg"] = df["prior_kd"] / df["prior_kd_count"].replace(0, np.nan)
df[:10][:3]


,event_id,event_date,fight_id,fighter_id,fighter_name,weight_class,outcome,winner_id,kd,ss_l,...,prior_kd,prior_ss_l,prior_ss_a,prior_td_l,prior_td_a,prior_ctrl_s,prior_kd_count,ss%,td%,kd_avg
0,296,2019-10-18,3570,1,Brendan Allen,Middleweight Bout,winner,1.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN
1,283,2020-02-29,3407,1,Brendan Allen,Middleweight Bout,winner,1.0,0.0,20.0,...,0.0,10.0,22.0,2.0,3.0,174.0,1,0.454545,0.666667,0.0
2,273,2020-06-27,3296,1,Brendan Allen,Middleweight Bout,winner,1.0,1.0,41.0,...,0.0,30.0,51.0,2.0,4.0,427.0,2,0.588235,0.500000,0.0


In [9]:
df.loc[
    df["fighter_name"].str.casefold().eq("song yadong")
].sort_values("event_date", ascending=False).head(1)

,event_id,event_date,fight_id,fighter_id,fighter_name,weight_class,outcome,winner_id,kd,ss_l,...,prior_kd,prior_ss_l,prior_ss_a,prior_td_l,prior_td_a,prior_ctrl_s,prior_kd_count,ss%,td%,kd_avg
1374,7,2026-08-29,78,156,Song Yadong,Bantamweight Bout,winner,156.0,1.0,11.0,...,7.0,958.0,2194.0,12.0,29.0,1767.0,17,0.436645,0.413793,0.411765


In [10]:
# One row per decisive matchup; fighter 1 is picked by a seeded random swap.
feature_cols = ["ss%", "td%", "kd_avg", "prior_wr"]
keys = ["event_id", "event_date", "fight_id"]

rows = df.loc[
    df["decisive"],
    keys + ["fighter_id", "fighter_name", "won"] + feature_cols,
].sort_values(["fight_id", "fighter_id"]).copy()
rows["side"] = rows.groupby("fight_id").cumcount()

#randomly swap sides in ~half the fights so fighter 1 isn't always the lower ID
fight_ids = rows["fight_id"].unique()
swap_ids = fight_ids[np.random.default_rng(42).random(len(fight_ids)) < 0.5]
swap = rows["fight_id"].isin(swap_ids)
rows.loc[swap, "side"] = 1 - rows.loc[swap, "side"]

matches = (
    rows.loc[rows["side"].eq(0)]
    .merge(
        rows.loc[rows["side"].eq(1)],
        on=keys,
        suffixes=("_f1", "_f2"),
        validate="one_to_one",
    )
    .sort_values(["event_date", "fight_id"])
    .reset_index(drop=True)
)

X = pd.DataFrame({
    f"{col}_diff": matches[f"{col}_f1"] - matches[f"{col}_f2"]
    for col in feature_cols
})
y = matches["won_f1"].astype(int)  # 1 = fighter 1 wins; 0 = fighter 2 wins. 
assert 0.45 < y.mean() < 0.55, f"side swap broken: y mean {y.mean():.3f}"
y.value_counts().sort_index()
#count 1s and 0s here


won_f1
0    4515
1    4263
Name: count, dtype: int64

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# train = matches["event_date"] < pd.Timestamp("2024-01-01")
#make train 70, val 20, test 10 by event date
dates = matches["event_date"]
val_start, test_start = dates.quantile([0.7, 0.9])

train = dates < val_start
val = (dates < test_start) & (dates >=val_start)
test = dates >=test_start

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
model.fit(X.loc[train], y.loc[train])




,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['ss%_diff','td%_diff','kd_avg_diff','prior_wr_diff']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data

In [12]:
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score, confusion_matrix

val_probs = model.predict_proba(X.loc[val])[:, 1]
y_val = y.loc[val]
pred_val = (val_probs >=0.5).astype(int)
print("accuracy score = ", accuracy_score(y_val, pred_val))
print("log loss = ", log_loss(y_val, val_probs))
print("brier:   ", brier_score_loss(y_val, val_probs))
print("roc auc: ", roc_auc_score(y_val, val_probs))
print(confusion_matrix(y_val, pred_val))

accuracy score =  0.5764840182648402
log loss =  0.6796450340354461
brier:    0.24330791519337822
roc auc:  0.612621736319653
[[703 185]
 [557 307]]


In [13]:
y.loc[val].value_counts(normalize=True)

won_f1
0    0.506849
1    0.493151
Name: proportion, dtype: float64

In [ ]:
# song_umar prediction
song_umar = matches.loc[matches["fight_id"].eq(78)]
if len(song_umar) != 1:
    raise ValueError("Expected exactly one matchup with fight_id 78")

song_umar_probs = model.predict_proba(X.loc[song_umar.index])
# Probability columns follow model.classes_, not the fighter order.
class_probs = dict(zip(model.classes_, song_umar_probs[0]))
fight = song_umar.iloc[0]

prediction = pd.DataFrame({
    "fighter_name": [fight["fighter_name_f1"], fight["fighter_name_f2"]],
    "win_probability": [class_probs[1], class_probs[0]],
})
prediction

,fighter_name,win_probability
0,Song Yadong,0.442421
1,Umar Nurmagomedov,0.557579
